# California House Price Prediction

**Goal:** predict the price of houses in an area of California from information about the area (income, location, house age, population and so on).

This is a regression problem: we predict a number, not a category.

**Pipeline:** load data → explore → split → train models → check results → which clues matter

Data: the California housing file that Google Colab provides in its `sample_data` folder (no upload needed).

## Step 1: Load the data

In [ ]:
import pandas as pd

df = pd.read_csv("/content/sample_data/california_housing_train.csv")
df.head()

## Step 2: Explore the data
Size, column types and missing values.

In [ ]:
print(df.shape)
print()
df.info()

17,000 rows and 9 columns, no missing values, and every column is already a number, so no encoding is needed. The last column, `median_house_value`, is the price we predict.

In [ ]:
df.describe()

The 50% row is the median (the middle value). The maximum price is 500,001 and the 75% value is 265,000: prices above 500,000 were all recorded as 500,001, so the price is capped.

## Step 3: Split into clues (X), answer (y), and train/test sets
80% of the areas teach the model, 20% are held back as a test.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('median_house_value', axis=1)   # clues
y = df['median_house_value']                # the price we predict

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)

## Step 4: Model 1, Linear Regression (baseline)

Two ways to measure a regression model:
- **Average error:** on average, how many dollars the guesses are away from the real price (smaller is better).
- **R2 score:** how much better the model is than always guessing the average price. 1 is perfect and 0 means no better than the average guess.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Average error:", mean_absolute_error(y_test, predictions))
print("R2 score:", r2_score(y_test, predictions))

Check against the lazy model that always guesses the average training price:

In [ ]:
import numpy as np

average_guess = np.full(len(y_test), y_train.mean())
print("Error if we always guess the average:", mean_absolute_error(y_test, average_guess))

## Step 5: Model 2, Random Forest
Many decision trees that vote. It can learn patterns that are not a straight line, such as how much location matters.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

print("Average error:", mean_absolute_error(y_test, rf_predictions))
print("R2 score:", r2_score(y_test, rf_predictions))

The Random Forest is clearly better: its average error is about 32,000 (against about 50,000) and its R2 is about 0.82 (against 0.66). Here the more complex model wins, unlike in the churn project, so the lesson is to compare against a simple baseline and let the numbers decide.

## Step 6: Which clues matter most?
The numbers add up to 1, and a bigger number means the model relied on that clue more.

In [ ]:
importance = pd.Series(rf_model.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False))

Income (about 0.52) and location (longitude and latitude, about 0.32 together) matter most. The rest matter little. This shows what the model used, not what causes high prices.

## Limitations

- The prices are capped at 500,001, so the model can never predict a price above about 500,000.
- The data is from 1990, so the prices are old. It was used to practise the method.